# Hen Pose Training — YOLO12s Pose (From Scratch)

Trains a **YOLO12s-pose** model from scratch on the merged+augmented hen pose dataset.

This notebook:
- builds a local YOLO12s pose config from the shipped YOLO12 detect config,
- initializes the model with random weights,
- trains with `pretrained=False` (no weight warm-start).

Run the cells top to bottom.

## 1. Setup


In [1]:
from pathlib import Path
import random
import shutil
import subprocess
import sys

import yaml
import cv2
import numpy as np
import matplotlib.pyplot as plt

import torch

try:
    import ultralytics
    from ultralytics import YOLO
except ModuleNotFoundError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "ultralytics"])
    import ultralytics
    from ultralytics import YOLO

ROOT = Path("..").resolve()
DATA_YAML = ROOT / "dataset/pose/pose-dataset-merged-aug-split/data.yaml"
MODELS_DIR = ROOT / "models"

BASE_DETECT_CFG = ROOT / "ultralytics/cfg/models/v12/yolov12.yaml"
MODEL_CFG = ROOT / "config/yolov12s-pose-hen.yaml"
RUN_NAME = "hen_pose_yolo12s_split_v2"

# 12GB-safe defaults.
IMG_SIZE = 640
EPOCHS = 220
PATIENCE = 50
BATCH = 12

assert DATA_YAML.exists(), f"Missing dataset yaml: {DATA_YAML}"
assert BASE_DETECT_CFG.exists(), f"Missing base YOLO12 detect cfg: {BASE_DETECT_CFG}"
MODELS_DIR.mkdir(parents=True, exist_ok=True)
MODEL_CFG.parent.mkdir(parents=True, exist_ok=True)

# Generate a YOLO12 pose cfg (s-scale via filename) from the local detect cfg.
base_text = BASE_DETECT_CFG.read_text(encoding="utf-8")
if "kpt_shape:" not in base_text:
    base_text = base_text.replace(
        "nc: 80 # number of classes",
        "nc: 80 # number of classes\nkpt_shape: [10, 3] # [num_keypoints, dims]",
    )
base_text = base_text.replace(
    "  - [[14, 17, 20], 1, Detect, [nc]] # Detect(P3, P4, P5)",
    "  - [[14, 17, 20], 1, Pose, [nc, kpt_shape]] # Pose(P3, P4, P5)",
)
MODEL_CFG.write_text(base_text, encoding="utf-8")

print("DATA_YAML:", DATA_YAML)
print("BASE_DETECT_CFG:", BASE_DETECT_CFG)
print("MODEL_CFG:", MODEL_CFG)
print("RUN_NAME:", RUN_NAME)
print("IMG_SIZE:", IMG_SIZE, "BATCH:", BATCH)

ultralytics.checks()
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Ultralytics 8.4.66  Python-3.13.14 torch-2.12.0+cu132 CUDA:0 (NVIDIA GeForce RTX 5070, 12227MiB)
Setup complete  (12 CPUs, 31.1 GB RAM, 240.4/464.8 GB disk)
Torch: 2.12.0+cu132
CUDA available: True
GPU: NVIDIA GeForce RTX 5070
VRAM: 12.8 GB


## 2. Sanity-check the dataset

Confirms the YAML is a pose dataset (has `kpt_shape`) and that images and labels line up.


In [2]:
cfg = yaml.safe_load(DATA_YAML.read_text(encoding="utf-8"))
root = Path(cfg.get("path", DATA_YAML.parent))
if not root.is_absolute():
    root = (DATA_YAML.parent / root).resolve()


def count(split_key):
    img_dir = root / cfg[split_key]
    lbl_dir = Path(str(img_dir).replace("images", "labels"))
    imgs = [p for p in img_dir.glob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".webp"}]
    lbls = list(lbl_dir.glob("*.txt"))
    return img_dir, lbl_dir, imgs, lbls


def check_label_columns(label_paths, expected_cols=35, sample_limit=200):
    bad = []
    checked = 0
    for lp in label_paths[:sample_limit]:
        with open(lp, "r", encoding="utf-8") as f:
            for ln_i, line in enumerate(f, start=1):
                line = line.strip()
                if not line:
                    continue
                cols = len(line.split())
                checked += 1
                if cols != expected_cols:
                    bad.append((lp.name, ln_i, cols))
    return checked, bad


assert "kpt_shape" in cfg, "data.yaml has no kpt_shape — this is not a pose dataset YAML"
assert cfg["kpt_shape"] == [10, 3], f"Expected kpt_shape [10, 3], got {cfg['kpt_shape']}"

train_img_dir, train_lbl_dir, train_imgs, train_lbls = count("train")
val_img_dir, val_lbl_dir, val_imgs, val_lbls = count("val")

checked_train, bad_train = check_label_columns(train_lbls)
checked_val, bad_val = check_label_columns(val_lbls)

print(f"Train images/labels: {len(train_imgs)} / {len(train_lbls)}")
print(f"Val   images/labels: {len(val_imgs)} / {len(val_lbls)}")
print(f"kpt_shape : {cfg['kpt_shape']}")
print(f"flip_idx  : {cfg.get('flip_idx')}")
print(f"classes   : {cfg['names']}")
print(f"Checked train label rows: {checked_train}")
print(f"Checked val   label rows: {checked_val}")

if bad_train or bad_val:
    print("\nFound bad label rows (expected 35 columns):")
    for rec in (bad_train + bad_val)[:10]:
        print("  file={}, line={}, cols={}".format(*rec))
    raise ValueError("Dataset has malformed pose labels. Fix before training.")
else:
    print("All checked label rows have 35 columns.")

Train images/labels: 422 / 422
Val   images/labels: 23 / 23
kpt_shape : [10, 3]
flip_idx  : [0, 1, 3, 2, 5, 4, 6, 7, 8, 9]
classes   : {0: 'hen'}
Checked train label rows: 556
Checked val   label rows: 61
All checked label rows have 35 columns.


## 3. Build YOLO12s-pose model config and initialize from scratch

Ultralytics in this workspace ships `yolov12.yaml` (detect) but not a YOLO12 pose YAML.
This cell generates `config/yolov12s-pose-hen.yaml` by replacing the final head with `Pose` and adding `kpt_shape`.
Then it initializes `YOLO(MODEL_CFG, task="pose")` with random weights.

In [3]:
model = YOLO(str(MODEL_CFG), task="pose")

# Guard: this must be a pose model and must include keypoint shape.
model_yaml = getattr(model.model, "yaml", {})
if model.task != "pose" or "kpt_shape" not in model_yaml:
    raise RuntimeError(
        f"'{MODEL_CFG}' is not a pose config (task={model.task}, "
        f"kpt_shape present={'kpt_shape' in model_yaml})."
    )

print(f"Built from scratch cfg: {MODEL_CFG}")
print(f"task={model.task} | model kpt_shape={model_yaml['kpt_shape']}")
print("Model initialized with random weights (no pretrained checkpoint loaded).")

Built from scratch cfg: C:\dev\poultry-vision\config\yolov12s-pose-hen.yaml
task=pose | model kpt_shape=[10, 3]
Model initialized with random weights (no pretrained checkpoint loaded).


## 4. Train (from scratch)

Configured for 12GB VRAM with a true scratch run:
- model built from local pose YAML
- `pretrained=False`
- `imgsz=640`, `batch=12`
- AdamW with moderate augmentation

In [4]:
results = model.train(
    data=str(DATA_YAML),

    # Training
    epochs=EPOCHS,
    patience=PATIENCE,
    batch=BATCH,
    imgsz=IMG_SIZE,
    pretrained=False,

    # Hardware
    device=0 if torch.cuda.is_available() else "cpu",
    workers=8,
    amp=True,
    cache=True,

    # Optimization
    optimizer="AdamW",
    lr0=0.001,
    lrf=0.01,
    weight_decay=0.0005,

    # Augmentation
    # Offline augmentation already supplies affine/color variation. Keep
    # online geometry moderate so precise keypoint locations remain learnable.
    degrees=8.0,
    translate=0.05,
    scale=0.25,
    shear=2.0,
    perspective=0.0,
    fliplr=0.5,
    flipud=0.0,
    hsv_h=0.015,
    hsv_s=0.5,
    hsv_v=0.3,
    mosaic=0.25,
    mixup=0.0,
    copy_paste=0.0,

    # Validation/outputs
    val=True,
    plots=True,
    save=True,
    project="runs/pose",
    name=RUN_NAME,
    exist_ok=True,
    verbose=True,
    seed=42,
    deterministic=True,
)

print("Run dir:", results.save_dir)

Ultralytics 8.4.66  Python-3.13.14 torch-2.12.0+cu132 CUDA:0 (NVIDIA GeForce RTX 5070, 12227MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=12, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\dev\poultry-vision\dataset\pose\pose-dataset-merged-aug-split\data.yaml, degrees=8.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=220, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.5, hsv_v=0.3, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=C:\dev\poultry-vision\config\yolov12s-pose-hen.yaml, momentum=0.937, mosaic=0.25, multi_scale=0.0, name=hen_

c:\dev\poultry-vision\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))
val: Fast image access  (ping: 0.00.0 ms, read: 442.579.9 MB/s, size: 494.2 KB)
val: Scanning C:\dev\poultry-vision\dataset\pose\pose-dataset-merged-aug-split\labels\val.cache... 23 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 23/23 8.8Mit/s 0.0s
WARNING cache='ram' may produce non-deterministic training results. Consider cache='disk' as a deterministic alternative if your disk space allows.
val: Caching images (0.0GB RAM): 100% ━━━━━━━━━━━━ 23/23 710.0it/s 0.0s
optimizer: AdamW(lr=0.001, momentum=0.937) with parameter groups 119 weight(decay=0.0), 129 weight(decay=0.00046875), 128 bias(decay=0.0)
Plotting labels to C:\dev\poultry-vision\runs\pose\runs\pose\hen_pose_yolo12s_split_v2\labels.jpg... 
Image sizes 640 train, 640 val
Using 8 dataloader workers
Logging result

## 5. Validate the best weights


In [5]:
best_weights = Path(results.save_dir) / "weights" / "best.pt"
print("Best weights:", best_weights, "| exists:", best_weights.exists())

best = YOLO(str(best_weights))
metrics = best.val(data=str(DATA_YAML), split="val", imgsz=IMG_SIZE, plots=True)
print("\n=== Final validation metrics ===")
print(f"Box  mAP@0.5      : {metrics.box.map50:.4f}")
print(f"Box  mAP@0.5:0.95 : {metrics.box.map:.4f}")
print(f"Pose mAP@0.5      : {metrics.pose.map50:.4f}")
print(f"Pose mAP@0.5:0.95 : {metrics.pose.map:.4f}")


Best weights: C:\dev\poultry-vision\runs\pose\runs\pose\hen_pose_yolo12s_split_v2\weights\best.pt | exists: True
Ultralytics 8.4.66  Python-3.13.14 torch-2.12.0+cu132 CUDA:0 (NVIDIA GeForce RTX 5070, 12227MiB)
YOLOv12s-pose-hen summary (fused): 168 layers, 9,400,317 parameters, 0 gradients, 20.3 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 610.2150.1 MB/s, size: 438.9 KB)
val: Scanning C:\dev\poultry-vision\dataset\pose\pose-dataset-merged-aug-split\labels\val.cache... 23 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 23/23 10.7Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Pose(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 1.1s/it 2.3s6.9s
                   all         23         61      0.914      0.902      0.919       0.57       0.88      0.869      0.856      0.436
Speed: 2.0ms preprocess, 9.7ms inference, 0.0ms loss, 0.8ms postprocess per image
Results saved to C:\dev\poultry-vision\runs\pose\val

## 6. Final evaluation on the untouched test split

Use validation for checkpoint selection and tuning. Run this test evaluation once after training choices are finalized; do not use test results to tune hyperparameters.

In [6]:
assert cfg.get("test"), "data.yaml must define an untouched test split"
test_metrics = best.val(data=str(DATA_YAML), split="test", imgsz=IMG_SIZE, plots=True)
print("\n=== Final held-out test metrics ===")
print(f"Box  precision     : {test_metrics.box.p.mean():.4f}")
print(f"Box  recall        : {test_metrics.box.r.mean():.4f}")
print(f"Box  mAP@0.5       : {test_metrics.box.map50:.4f}")
print(f"Box  mAP@0.5:0.95  : {test_metrics.box.map:.4f}")
print(f"Pose mAP@0.5       : {test_metrics.pose.map50:.4f}")
print(f"Pose mAP@0.5:0.95  : {test_metrics.pose.map:.4f}")

Ultralytics 8.4.66  Python-3.13.14 torch-2.12.0+cu132 CUDA:0 (NVIDIA GeForce RTX 5070, 12227MiB)
val: Fast image access  (ping: 0.10.1 ms, read: 646.071.7 MB/s, size: 518.0 KB)
val: Scanning C:\dev\poultry-vision\dataset\pose\pose-dataset-merged-aug-split\labels\test.cache... 23 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 23/23 10.7Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Pose(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 1.1it/s 1.8s5.8s
                   all         23        100      0.882        0.7      0.718       0.32       0.78       0.56      0.567      0.193
Speed: 2.0ms preprocess, 12.5ms inference, 0.0ms loss, 0.6ms postprocess per image
Results saved to C:\dev\poultry-vision\runs\pose\val-4

=== Final held-out test metrics ===
Box  precision     : 0.8821
Box  recall        : 0.7000
Box  mAP@0.5       : 0.7176
Box  mAP@0.5:0.95  : 0.3201
Pose mAP@0.5       : 0.5674
Pose mAP@0.5:0.95  : 0.193

## 7. Save best model into `models/`


In [ ]:
out_path = MODELS_DIR / f"{RUN_NAME}.pt"
shutil.copy2(best_weights, out_path)
print("Saved:", out_path)

Saved: C:\dev\poultry-vision\models\hen_pose_yolo12s_scratch.pt


## 8. Visual check on untouched test images


In [ ]:
test_img_dir = root / cfg["test"]
test_imgs = [p for p in test_img_dir.glob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".webp"}]
samples = random.sample(test_imgs, min(8, len(test_imgs)))
preds = best.predict(source=samples, conf=0.25, imgsz=IMG_SIZE, save=False, verbose=False)

cols = 4
rows = int(np.ceil(len(preds) / cols))
fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 4.5 * rows))
axes = np.array(axes).reshape(-1)
for ax in axes[len(preds):]:
    ax.axis("off")
for ax, r in zip(axes, preds):
    ax.imshow(cv2.cvtColor(r.plot(), cv2.COLOR_BGR2RGB))
    ax.set_title(f"{Path(r.path).name} | {len(r.boxes)} hen")
    ax.axis("off")
plt.tight_layout()
plt.show()

<Figure size 2000x900 with 8 Axes>